## Web Scraping Property Listings - Mozambique
# Project Overview
This notebook demonstrates web scraping of property listings from Property24 Mozambique, specifically targeting properties for sale in Mozambique in all the provinces. The scraped data is saved as structured datasets for further analysis.

# Key Components

Requests: For making HTTP requests to the website

BeautifulSoup: For parsing HTML and extracting data

Pandas: For data manipulation and storage

Time: For implementing polite scraping delays

# HTML Classes Used
- p24_regularTile: Main property container

- p24_propertyTitle: Property title

- p24_price: Price information

- p24_location: Location name

- p24_address: Full address

- p24_excerpt: Property description

- p24_featureDetails: Feature containers (bedrooms, bathrooms, parking)

- p24_size: Floor size information

In [16]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [17]:
html_content = "https://www.property24.co.mz/property-for-sale-in-maputo%20city-c2279"

In [18]:
soup = BeautifulSoup(html_content, 'html.parser')

C:\Users\ACER\AppData\Local\Temp\ipykernel_4188\1342470912.py:1: MarkupResemblesLocatorWarning: The input looks more like a URL than markup. You may want to use an HTTP client like requests to get the document behind the URL, and feed that document to Beautiful Soup.
  soup = BeautifulSoup(html_content, 'html.parser')


In [19]:
url = "https://www.property24.co.mz/property-for-sale-in-maputo%20city-c2279"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}


In [20]:
response = requests.get(url, headers=headers)

In [26]:
if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Find all property listings
    listings = soup.find_all('div', class_='p24_regularTile')
    
    data = []
    
    for listing in listings:
        # Extract title
        title_tag = listing.find('span', class_='p24_propertyTitle')
        title = title_tag.get_text(strip=True) if title_tag else None
        
        # Extract price
        price_tag = listing.find('span', class_='p24_price')
        price = price_tag.get_text(strip=True) if price_tag else None
        
        # Extract location
        location_tag = listing.find('span', class_='p24_location')
        location = location_tag.get_text(strip=True) if location_tag else None
        
        # Extract bedrooms
        bed_span = listing.find(
            'span',
            class_='p24_featureDetails',
            title='Bedrooms'
        )
        beds = bed_span.find('span').get_text(strip=True) if bed_span else None
        
        # Extract bathrooms
        bath_span = listing.find(
            'span',
            class_='p24_featureDetails',
            title='Bathrooms'
        )
        baths = bath_span.find('span').get_text(strip=True) if bath_span else None
        
        # Extract parking
        parking_span = listing.find(
            'span',
            class_='p24_featureDetails',
            title='Parking Spaces'
        )
        parking = parking_span.find('span').get_text(strip=True) if parking_span else None
        
        # Extract URL
        link_tag = listing.find('a', href=True)
        url_full = link_tag['href'] if link_tag else None
        
        if url_full and not url_full.startswith('http'):
            url_full = 'https://www.property24.co.mz' + url_full
        
        # Add listing to data
        data.append({
            'Title': title,
            'Price': price,
            'Location': location,
            'Bedrooms': beds,
            'Bathrooms': baths,
            'Parking': parking,
            'URL': url_full
        })

    # Create DataFrame
    df = pd.DataFrame(data)
    print(df.head())
    
    # Save to CSV
    df.to_csv(
        'property_listings.csv',
        index=False,
        encoding='utf-8'
    )
    
    print(f"\nSaved {len(df)} listings to property_listings.csv")

else:
    print(f"Failed to retrieve page. Status code: {response.status_code}")

                        Title          Price          Location Bedrooms  \
0         Industrial Property  MT 65 000 000            Maputo     None   
1  3 Bedroom Apartment / Flat  MT 17 000 000  Polana Cimento A        3   
2  3 Bedroom Apartment / Flat  MT 18 000 000  Polana Cimento A        3   
3  3 Bedroom Apartment / Flat  MT 24 200 000     Sommerschield        3   
4  2 Bedroom Apartment / Flat  MT 14 500 000  Polana Cimento A        2   

  Bathrooms Parking                                                URL  
0      None    None  https://www.property24.co.mz/industrial-proper...  
1         3       1  https://www.property24.co.mz/3-bedroom-apartme...  
2         3    None  https://www.property24.co.mz/3-bedroom-apartme...  
3         3       2  https://www.property24.co.mz/3-bedroom-apartme...  
4         2       1  https://www.property24.co.mz/2-bedroom-apartme...  

Saved 21 listings to property_listings.csv


In [27]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Base URL with page parameter
#base_url = "https://www.property24.co.mz/property-for-sale-in-tete%20city-c2294?page={}"
base_url = "https://www.property24.co.mz/property-for-sale-in-marracuene-c2281"

#https://www.property24.co.mz/property-for-sale-in-matola-c2280
#https://www.property24.co.mz/property-for-sale-in-beira-c2295
#https://www.property24.co.mz/property-for-sale-in-nampula-c2291
#https://www.property24.co.mz/property-for-sale-in-chimoio-c2296
#https://www.property24.co.mz/property-for-sale-in-nacala-c2292
#https://www.property24.co.mz/property-for-sale-in-tete-c2294
#https://www.property24.co.mz/property-for-sale-in-bilene-c2274
#https://www.property24.co.mz/property-for-sale-in-vilanculos-c2283
#https://www.property24.co.mz/property-for-sale-in-inhambane%20city-c2284
#https://www.property24.co.mz/property-for-sale-in-marracuene-c2281
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Store all listings
all_data = []

# Determine total pages (from the HTML we saw, it's 44 pages)
# You can also extract this dynamically from the pagination element
total_pages = 10

print(f"Starting to scrape {total_pages} pages...")

for page in range(1, total_pages + 1):
    print(f"Scraping page {page}/{total_pages}...", end=" ")
    
    url = base_url.format(page)
    
    try:
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Find all property listings on this page
            listings = soup.find_all('div', class_='p24_regularTile')
            
            print(f"Found {len(listings)} listings")
            
            for listing in listings:
                # Extract title
                title_tag = listing.find('span', class_='p24_propertyTitle')
                title = title_tag.get_text(strip=True) if title_tag else None
                
                # Extract price
                price_tag = listing.find('span', class_='p24_price')
                price = price_tag.get_text(strip=True) if price_tag else None
                
                # Extract location
                location_tag = listing.find('span', class_='p24_location')
                location = location_tag.get_text(strip=True) if location_tag else None
                
                # Extract address
                address_tag = listing.find('span', class_='p24_address')
                address = address_tag.get_text(strip=True) if address_tag else None
                
                # Extract excerpt/description
                excerpt_tag = listing.find('span', class_='p24_excerpt')
                excerpt = excerpt_tag.get_text(strip=True) if excerpt_tag else None
                
                # Extract bedrooms
                bed_span = listing.find('span', class_='p24_featureDetails', title='Bedrooms')
                beds = bed_span.find('span').get_text(strip=True) if bed_span else None
                
                # Extract bathrooms
                bath_span = listing.find('span', class_='p24_featureDetails', title='Bathrooms')
                baths = bath_span.find('span').get_text(strip=True) if bath_span else None
                
                # Extract parking spaces
                parking_span = listing.find('span', class_='p24_featureDetails', title='Parking Spaces')
                parking = parking_span.find('span').get_text(strip=True) if parking_span else None
                
                # Extract floor size if available
                floor_size = listing.find('span', class_='p24_size')
                floor_size_text = floor_size.find('span').get_text(strip=True) if floor_size else None
                
                # Extract URL
                link_tag = listing.find('a', href=True)
                url_full = link_tag['href'] if link_tag else None
                if url_full and not url_full.startswith('http'):
                    url_full = 'https://www.property24.co.mz' + url_full
                
                all_data.append({
                    'Page': page,
                    'Title': title,
                    'Price': price,
                    'Location': location,
                    'Address': address,
                    'Description': excerpt,
                    'Bedrooms': beds,
                    'Bathrooms': baths,
                    'Parking': parking,
                    'Floor_Size_m2': floor_size_text,
                    'URL': url_full
                })
            
            # Be polite - wait 1 second between requests to avoid overwhelming the server
            time.sleep(1)
            
        else:
            print(f"Failed with status code: {response.status_code}")
            break
            
    except Exception as e:
        print(f"Error on page {page}: {e}")
        continue

# Create DataFrame with all data
df = pd.DataFrame(all_data)

print(f"\n{'='*50}")
print(f"Scraping complete!")
print(f"Total listings extracted: {len(df)}")
print(f"Total pages scraped: {df['Page'].nunique()}")
print(f"{'='*50}")

# Display first few rows
print("\nFirst 5 listings:")
print(df.head())

# Save to CSV
#df.to_csv('all_maputo_properties.csv', index=False, encoding='utf-8')
df.to_excel(r'C:\Users\ACER\Desktop\Output\marracuen.xlsx', index = False)
print(f"\nData saved to 'all_maputo_properties.csv'")

# Display some basic statistics
print("\nBasic Statistics:")
print(f"Properties with prices: {df['Price'].notna().sum()}")
print(f"Average price (first 10 valid entries): {df['Price'].head(10).tolist()}")

# Show unique locations
print(f"\nUnique locations: {df['Location'].nunique()}")
print(f"Locations: {df['Location'].unique()[:10]}")  # First 10 locations

Starting to scrape 10 pages...
Scraping page 1/10... Found 8 listings
Scraping page 2/10... Found 8 listings
Scraping page 3/10... Found 8 listings
Scraping page 4/10... Found 8 listings
Scraping page 5/10... Found 8 listings
Scraping page 6/10... Found 8 listings
Scraping page 7/10... Found 8 listings
Scraping page 8/10... Found 8 listings
Scraping page 9/10... Found 8 listings
Scraping page 10/10... Found 8 listings

Scraping complete!
Total listings extracted: 80
Total pages scraped: 10

First 5 listings:
   Page                Title          Price    Location  \
0     1   Vacant Land / Plot   MT 4 550 000  Marracuene   
1     1   Vacant Land / Plot   MT 4 500 000  Marracuene   
2     1                 Farm  MT 96 000 000  Marracuene   
3     1  Commercial Property  MT 27 000 000  Marracuene   
4     1   Vacant Land / Plot   MT 4 500 000  Marracuene   

                                      Address  \
0    Estrada Circular, Marracuene, Marracuene   
1  1 Marracuene Vista, Marracuene